In [1]:
import pandas as pd

In [18]:
df = pd.read_csv("atp_matches_2023.csv")
df

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2023-9900,United Cup,Hard,18,A,20230102,300,126203,3.0,NaN,...,62.0,47.0,15.0,12.0,9.0,9.0,9.0,3355.0,16.0,2375.0
1,2023-9900,United Cup,Hard,18,A,20230102,299,126207,NaN,NaN,...,12.0,8.0,3.0,4.0,1.0,3.0,19.0,2000.0,23.0,1865.0
2,2023-9900,United Cup,Hard,18,A,20230102,296,126203,3.0,NaN,...,62.0,51.0,7.0,12.0,2.0,2.0,9.0,3355.0,10.0,2905.0
3,2023-9900,United Cup,Hard,18,A,20230102,295,126207,NaN,NaN,...,41.0,26.0,12.0,9.0,6.0,9.0,19.0,2000.0,245.0,220.0
4,2023-9900,United Cup,Hard,18,A,20230102,292,126774,1.0,NaN,...,58.0,48.0,18.0,16.0,1.0,2.0,4.0,5550.0,16.0,2375.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2981,2023-M-DC-2023-WG2-PO-RSA-LUX-01,Davis Cup WG2 PO: RSA vs LUX,NaN,4,D,20230204,5,202335,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1717.0,1.0
2982,2023-M-DC-2023-WG2-PO-TUN-CYP-01,Davis Cup WG2 PO: TUN vs CYP,NaN,4,D,20230203,1,117365,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,990.0,11.0,279.0,190.0
2983,2023-M-DC-2023-WG2-PO-TUN-CYP-01,Davis Cup WG2 PO: TUN vs CYP,NaN,4,D,20230203,2,121411,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,364.0,131.0,894.0,15.0
2984,2023-M-DC-2023-WG2-PO-TUN-CYP-01,Davis Cup WG2 PO: TUN vs CYP,NaN,4,D,20230203,4,144949,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,894.0,15.0,285.0,184.0


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2986 entries, 0 to 2985
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   tourney_id          2986 non-null   object 
 1   tourney_name        2986 non-null   object 
 2   surface             2933 non-null   object 
 3   draw_size           2986 non-null   int64  
 4   tourney_level       2986 non-null   object 
 5   tourney_date        2986 non-null   int64  
 6   match_num           2986 non-null   int64  
 7   winner_id           2986 non-null   int64  
 8   winner_seed         1250 non-null   float64
 9   winner_entry        473 non-null    object 
 10  winner_name         2986 non-null   object 
 11  winner_hand         2986 non-null   object 
 12  winner_ht           2969 non-null   float64
 13  winner_ioc          2986 non-null   object 
 14  winner_age          2986 non-null   float64
 15  loser_id            2986 non-null   int64  
 16  loser_

In [20]:
# Step 1: Create winner and loser rows
winner_df = df[['winner_name', 'surface']].copy()
winner_df['result'] = 1  # Win

loser_df = df[['loser_name', 'surface']].copy()
loser_df['result'] = 0  # Loss

# Step 2: Rename to common format
winner_df = winner_df.rename(columns={'winner_name': 'player_name'})
loser_df = loser_df.rename(columns={'loser_name': 'player_name'})

# Step 3: Combine
player_match_df = pd.concat([winner_df, loser_df], ignore_index=True)


In [21]:
player_match_df

,player_name,surface,result
0,Taylor Fritz,Hard,1
1,Frances Tiafoe,Hard,1
2,Taylor Fritz,Hard,1
3,Frances Tiafoe,Hard,1
4,Stefanos Tsitsipas,Hard,1
...,...,...,...
5967,Devin Badenhorst,NaN,0
5968,Skander Mansouri,NaN,0
5969,Menelaos Efstathiou,NaN,0
5970,Aziz Dougaz,NaN,0


In [22]:
player_summary = (
    player_match_df
    .groupby('player_name')
    .agg(
        total_matches=('result', 'count'),
        total_wins=('result', 'sum')
    )
    .reset_index()
)

player_summary['win_pct'] = player_summary['total_wins'] / player_summary['total_matches']


In [23]:
# Only keep wins
wins_df = player_match_df[player_match_df['result'] == 1]

# Count wins by player and surface
surface_wins = (
    wins_df
    .groupby(['player_name', 'surface'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)


In [24]:
player_surface_summary = player_summary.merge(surface_wins, on='player_name', how='left').fillna(0)


In [25]:
# Assume surfaces are 'Clay', 'Grass', 'Hard'
player_surface_summary['clay_advantage'] = player_surface_summary['Clay'] / player_surface_summary['total_wins']
player_surface_summary['grass_advantage'] = player_surface_summary['Grass'] / player_surface_summary['total_wins']
player_surface_summary['hard_advantage'] = player_surface_summary['Hard'] / player_surface_summary['total_wins']


In [26]:
player_surface_summary

,player_name,total_matches,total_wins,win_pct,Clay,Grass,Hard,clay_advantage,grass_advantage,hard_advantage
0,Abedallah Shelbayh,11,3,0.272727,1.0,0.0,2.0,0.333333,0.000000,0.666667
1,Adam Moundir,2,0,0.000000,0.0,0.0,0.0,NaN,NaN,NaN
2,Adrian Mannarino,67,43,0.641791,1.0,12.0,30.0,0.023256,0.279070,0.697674
3,Aisam Ul Haq Qureshi,1,1,1.000000,0.0,1.0,0.0,0.000000,1.000000,0.000000
4,Albert Ramos,38,12,0.315789,11.0,0.0,1.0,0.916667,0.000000,0.083333
...,...,...,...,...,...,...,...,...,...,...
435,Zhe Li,1,0,0.000000,0.0,0.0,0.0,NaN,NaN,NaN
436,Zhizhen Zhang,36,19,0.527778,10.0,2.0,7.0,0.526316,0.105263,0.368421
437,Zizou Bergs,12,3,0.250000,2.0,0.0,1.0,0.666667,0.000000,0.333333
438,Zsombor Piros,4,2,0.500000,1.0,0.0,1.0,0.500000,0.000000,0.500000


In [27]:
player_surface_summary.to_csv("player_surface_summary.csv")